In [2]:
! conda env list

# conda environments:
#
                         /home/sfgary/.miniconda
                         /home/sfgary/.miniconda/envs/mlflow
base                  *  /home/sfgary/pw/software/.miniconda3c



In [3]:
import pandas as pd
import numpy as np

# URLs
CSV_URL = "https://raw.githubusercontent.com/parallelworks/dynamic-learning-rivers/refs/heads/Nov-2023-log10-gss/scripts/prep_06_output_final_train.csv"
IXY_URL = "https://raw.githubusercontent.com/parallelworks/dynamic-learning-rivers/refs/heads/Nov-2023-log10-gss/scripts/prep_06_output_final_train.ixy"

# Read both files (both have headers; data rows align one-to-one)
features = pd.read_csv(CSV_URL)
meta = pd.read_csv(IXY_URL)

# Sanity check: rows must match up exactly
if len(features) != len(meta):
    raise ValueError(f"Row count mismatch: features={len(features)}, meta={len(meta)}")
    
# Add log10(respiraiton_rate)
features['log10_RR'] = np.log10(features["Normalized_Respiration_Rate_mg_DO_per_H_per_L_sediment"].abs())

# Attach group ID by position (reset indices to guarantee positional alignment)
features = features.reset_index(drop=True)
meta = meta.reset_index(drop=True)
features["gid"] = meta["gid"].values

# Group by gid and compute mean / std of each feature
grouped = features.groupby("gid")
means = grouped.mean(numeric_only=True)
stds = grouped.std(numeric_only=True)  # sample std (ddof=1); use ddof=0 for population std

# Save results
means.to_csv("group_feature_means.csv")
stds.to_csv("group_feature_stds.csv")

print(f"Processed {len(features)} rows across {grouped.ngroups} groups.")
print("Wrote: group_feature_means.csv, group_feature_stds.csv")

Processed 690 rows across 214 groups.
Wrote: group_feature_means.csv, group_feature_stds.csv


In [4]:
# The .sum() adds up the stds and means over all rows.
# For stds.sum(), this means add up all the standard deviations
# over all the groups. Note that it is small because for many
# groups, but not all, these within-group variability of features
# is zero.
# Compare the stds.sum() to the means.sum() - basically, how
# big is the total within-group variability compared to the
# total within-group average values?
stds.sum()/means.sum()

RA_SO                                                     0.005188
RA_dm                                                     0.003593
run_mm_cyr                                                0.002649
dor_pc_pva                                                0.004429
gwt_cm_cav                                                0.014353
ele_mt_cav                                                0.007633
slp_dg_cav                                                0.009485
sgr_dk_rav                                                0.084985
tmp_dc_cyr                                                0.002609
tmp_dc_cdi                                                0.001291
pre_mm_cyr                                                0.002726
pre_mm_cdi                                                0.005261
for_pc_cse                                                0.006778
crp_pc_cse                                                0.000832
pst_pc_cse                                                0.01

In [5]:
# Also, try this first element-by-element and then adding up
stds_over_means = stds/means
stds_over_means.sum()

RA_SO                                                      1.409260
RA_dm                                                      1.558355
run_mm_cyr                                                 0.930991
dor_pc_pva                                                 0.866025
gwt_cm_cav                                                 2.246500
ele_mt_cav                                                 1.808017
slp_dg_cav                                                 1.587688
sgr_dk_rav                                                 4.088420
tmp_dc_cyr                                                 0.198457
tmp_dc_cdi                                                 0.312819
pre_mm_cyr                                                 0.806499
pre_mm_cdi                                                 0.953006
for_pc_cse                                                 1.156735
crp_pc_cse                                                 1.465870
pst_pc_cse                                      

In both cases, the within group variabilty of the respiration rate is much higher than comparatively scaled quantities for all features.

In [6]:
# What does it look like in absolute terms (i.e. unscaled)?
stds.sum()

RA_SO                                                         3.464102
RA_dm                                                         0.907789
run_mm_cyr                                                  225.166605
dor_pc_pva                                                  215.929001
gwt_cm_cav                                                 1000.548017
ele_mt_cav                                                 1085.995856
slp_dg_cav                                                  127.017059
sgr_dk_rav                                                 2331.340387
tmp_dc_cyr                                                   57.735027
tmp_dc_cdi                                                   60.044428
pre_mm_cyr                                                  514.996440
pre_mm_cdi                                                   93.530744
for_pc_cse                                                   84.870490
crp_pc_cse                                                    2.886751
pst_pc

In [7]:
# How does the average within-group variability compare to the inter-group variability?
stds.mean()/means.std()

RA_SO                                                     0.008536
RA_dm                                                     0.002514
run_mm_cyr                                                0.003326
dor_pc_pva                                                0.001700
gwt_cm_cav                                                0.015908
ele_mt_cav                                                0.006548
slp_dg_cav                                                0.009796
sgr_dk_rav                                                0.039024
tmp_dc_cyr                                                0.004681
tmp_dc_cdi                                                0.005651
pre_mm_cyr                                                0.005559
pre_mm_cdi                                                0.006864
for_pc_cse                                                0.010051
crp_pc_cse                                                0.000608
pst_pc_cse                                                0.01

In [8]:
# Variability in the group variabilities versus variability in the mean group respiration rates
stds.std()/means.std()

RA_SO                                                     0.087472
RA_dm                                                     0.029306
run_mm_cyr                                                0.039308
dor_pc_pva                                                0.024698
gwt_cm_cav                                                0.124793
ele_mt_cav                                                0.047393
slp_dg_cav                                                0.074113
sgr_dk_rav                                                0.385780
tmp_dc_cyr                                                0.033118
tmp_dc_cdi                                                0.062458
pre_mm_cyr                                                0.061220
pre_mm_cdi                                                0.085683
for_pc_cse                                                0.099883
crp_pc_cse                                                0.005843
pst_pc_cse                                                0.12

In [9]:
# Variance in group variance compared to all possible variability
stds.std()/features.std()

Mean_DO_mg_per_L                                          0.086899
Mean_DO_percent_saturation                                0.028938
Mean_Temp_Deg_C                                           0.110817
Normalized_Respiration_Rate_mg_DO_per_H_per_L_sediment    0.339194
RA_SO                                                     0.086871
RA_dm                                                     0.030063
RA_ms_av                                                  0.094711
RA_ms_di                                                  0.119942
crp_pc_cse                                                0.005877
dor_pc_pva                                                0.025514
ele_mt_cav                                                0.048341
for_pc_cse                                                0.100531
gid                                                            NaN
gla_pc_cse                                                0.000000
gwt_cm_cav                                                0.12

In [10]:
# Average group variability compared to all possible variability
stds.mean()/features.std()

Mean_DO_mg_per_L                                          0.011181
Mean_DO_percent_saturation                                0.005182
Mean_Temp_Deg_C                                           0.015629
Normalized_Respiration_Rate_mg_DO_per_H_per_L_sediment    0.141494
RA_SO                                                     0.008478
RA_dm                                                     0.002579
RA_ms_av                                                  0.011418
RA_ms_di                                                  0.017555
crp_pc_cse                                                0.000612
dor_pc_pva                                                0.001756
ele_mt_cav                                                0.006679
for_pc_cse                                                0.010116
gid                                                            NaN
gla_pc_cse                                                0.000000
gwt_cm_cav                                                0.01